# BERTopic

#### Author information

- **Name:** Jaeseong Choe

- **email address:** cjssoote@gmail.com

- **GitHub:** https://github.com/sorrychoe

- **Personal Webpage:** https://jaeseongchoe.vercel.app/

## Part 1. Brief background of methodology

### Overview

- **BERTopic is a topic modeling technique that leverages transformer-based embeddings and a class-based TF-IDF (c-TF-IDF) approach** to generate dense clusters of documents, resulting in interpretable topics with meaningful descriptions.

### Situation Before BERTopic

- Traditional topic modeling methods like Latent Dirichlet Allocation (LDA) often struggled with capturing the semantic nuances of documents, especially when dealing with short texts or documents with overlapping topics.

### Why BERTopic Was Introduced

- BERTopic was introduced from the paper "BERTopic: Neural topic modeling with a class-based TF-IDF procedure" of Maarten Grootendorst. (2022).

- BERTopic was introduced to address the limitations of traditional models by incorporating transformer-based embeddings, which capture contextual information, and by utilizing c-TF-IDF for more coherent topic representations

### Use Cases

- BERTopic is applicable in various domains, including analyzing customer reviews, extracting themes from research articles, monitoring social media trends, and exploring historical document collections.

## Part 2. Key concept of methodology

### Key Concept

- BERTopic combines transformer-based embeddings with clustering algorithms and a class-based TF-IDF approach to identify and represent topics within a corpus. 
  

### Algorithmic Pipeline

- Unlike LDA, DTM, ToT, CTM, STM, and BTM, **BERTopic is not a Bayesian generative model**. It has no prior distribution over latent topics and no word-generating likelihood to invert, so there is no "generative story" to write down. Instead, BERTopic is a **deterministic embed-then-cluster pipeline** that produces topics as a post-hoc description of document clusters.

The pipeline involves four main steps:

**1. Document Embedding**

- Each document is converted into a vector representation using a pre-trained transformer model, such as BERT. 

**2. Dimensionality Reduction**

- The high-dimensional embeddings are reduced to a lower-dimensional space using techniques like UMAP to preserve semantic relationships. 

**3. Clustering**

- The reduced embeddings are clustered using algorithms like HDBSCAN to group semantically similar documents. 

**4. Topic Representation**

- For each cluster, a c-TF-IDF approach is applied to extract representative words, forming the topic description.

![BERTopic_Graphic](./img/bertopic_graphic.png)

### Mathematical Representation

- **Dimensionality Reduction (UMAP)**:
  UMAP embeds the high-dimensional embeddings $e_d \in \mathbb{R}^h$ into a low-dimensional space by constructing a fuzzy topological representation of the data in both spaces and minimizing the cross-entropy between them:
  $$
  C_{UMAP} = \sum_{(i,j)} \Big[ v_{ij} \log \frac{v_{ij}}{w_{ij}} + (1 - v_{ij}) \log \frac{1 - v_{ij}}{1 - w_{ij}} \Big]
  $$
  where $v_{ij}$ is the fuzzy-set membership strength between points $i,j$ in the original space and $w_{ij}$ is the corresponding strength in the reduced space.

- **Clustering (HDBSCAN)**:
  HDBSCAN builds a hierarchy over the reduced embeddings using the **mutual reachability distance**
  $$
  d_{mreach}(a,b) = \max\{\text{core}_k(a), \text{core}_k(b), d(a,b)\}
  $$
  and extracts the flat clustering that maximizes total cluster **stability**, leaving points with no stable cluster labeled as noise (topic $-1$). This step — not a hyperparameter chosen by the analyst — is what determines the number of topics $K$.

- **Class-based TF-IDF (c-TF-IDF)**:
  BERTopic's core contribution treats every cluster $c$ as a single concatenated document and scores term $t$ by:
  $$
  W_{t,c} = tf_{t,c} \cdot \ln\!\left(1 + \frac{A}{f_t}\right)
  $$
  where $tf_{t,c}$ is the frequency of term $t$ within cluster $c$, $f_t$ is the frequency of $t$ across all clusters, and $A$ is the average number of words per cluster ($A = \frac{1}{|C|}\sum_{c} |c|$). This is the class-level analogue of ordinary document-level TF-IDF, and it is what produces the ranked keyword list for each topic.

### Topic Assignment (in place of "Inference")

BERTopic has no posterior distribution to infer, so there is no Gibbs sampling or variational inference step as in the LDA family. What plays an analogous role is:

- **Hard assignment**: each document is assigned to the cluster label produced by HDBSCAN (or $-1$ if it is treated as noise).

- **Soft assignment (`probs`)**: `topic_model.fit_transform` can additionally return an approximate membership vector per document, derived from HDBSCAN's soft-clustering distances to cluster exemplars. This looks similar to LDA's $\theta_d$, but it is **not a Bayesian posterior** — it is a heuristic membership score computed after clustering, not a quantity the model was fit to optimize.

- **Topic merging/reduction**: because $K$ is a byproduct of HDBSCAN rather than a chosen hyperparameter, BERTopic additionally supports post-hoc topic reduction (merging the most similar c-TF-IDF vectors) to control the final topic count if needed.

### Strength

- BERTopic effectively captures semantic relationships in documents through transformer embeddings and provides coherent topic representations using c-TF-IDF. Its modular design allows for flexibility in choosing embedding models, dimensionality reduction techniques, and clustering algorithms, making it adaptable to various datasets and applications.

### Weakness

- Because the number of topics $K$ is determined by HDBSCAN rather than chosen via a perplexity/coherence search, BERTopic cannot be model-compared against LDA-family models on the same likelihood-based footing.

- Results are sensitive to the choice of embedding model, and to UMAP/HDBSCAN hyperparameters (`n_neighbors`, `min_dist`, `min_cluster_size`, `min_samples`); small changes can noticeably change the number and boundaries of topics.

- UMAP's stochastic optimization means that even with `random_state` fixed, results can vary slightly across library versions and hardware/thread counts.

- The noise topic ($-1$) can absorb a large fraction of documents when the corpus is noisy or the embedding model is not well matched to the domain/language, which reduces effective coverage of the "real" topics.

## Part 3. Example

### Precautions

- If you re-execute the code, there may be a slight difference in the result.

- Of course, the difference in the number or content of the topic will not be significant due to the learning rate, but the number of the topic changes.

In [1]:
# import librarys
import pandas as pd # for load excel data
import pyBigKinds as pbk # for preprocessing news data
from sklearn.feature_extraction.text import CountVectorizer # for vectorize text data 
from konlpy.tag import Mecab # for tokenize the Korean Words
from bertopic.vectorizers import ClassTfidfTransformer # for get c-TF-IDF value
from bertopic import BERTopic # for load the BERTopic model
from hdbscan import HDBSCAN # for tunning the BERTopic model
from umap import UMAP # for tunning the BERTopic model

# for ignore the warning message
import warnings
warnings.filterwarnings("ignore")

mecab = Mecab(dicpath='/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ko-dic')

In [2]:
def list_to_str(words: list):
    """Function that list data change to string for text preprocessing"""
    for i in range(len(words)):
        text = ""
        for word in words[i]:
            if text == "":
                text = word
            else: 
                text = text + " " + word
        words[i] = text
    return words

In [3]:
class CustomTokenizer:
    """ Define the Korean Tokenizer"""
    def __init__(self, tagger):
        self.tagger = tagger
    def __call__(self, sent):
        sent = sent[:1000000]
        word_tokens = self.tagger.morphs(sent)
        result = [word for word in word_tokens if len(word) > 1]
        return result

In [4]:
# data load
# The data is related to Handong University, 
# which was reported in major Korean daily newspapers from January 1995 to September 2024.
df = pd.read_excel("data/NewsResult_19950101-20240930.xlsx", engine="openpyxl")

# add the time stamp for BERTopic's topics_over_time
df["시점"] = (round(df["일자"]/10000,0)).astype(int)
df = df.sort_values("시점")
df.reset_index(drop=True, inplace=True)

# text Preprocessing 
words = pbk.keyword_parser(pbk.keyword_list(df))
words = list_to_str(words)
timestamp = df["시점"].tolist()

# Define tokenizer & Vectorizer
custom_tokenizer = CustomTokenizer(mecab)
vectorizer = CountVectorizer(tokenizer=custom_tokenizer, max_features=3000)

In [5]:
# set the HDBSCAN Model for tunning BERTopic model
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_samples=10, gen_min_span_tree=True, prediction_data=False, min_cluster_size=20)

# Define Topic Model for BERTopic
topic_model = BERTopic(
    embedding_model="sentence-transformers/xlm-r-100langs-bert-base-nli-stsb-mean-tokens", 
    vectorizer_model=vectorizer,
    hdbscan_model=hdbscan_model,
    ctfidf_model=ctfidf_model,
    top_n_words=20,
    min_topic_size=20,
    verbose=True
)

In [6]:
# fit the data
topics, probs = topic_model.fit_transform(words)

2026-09-16 18:24:28,472 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/252 [00:00<?, ?it/s]

2026-09-16 18:40:06,335 - BERTopic - Embedding - Completed ✓
2026-09-16 18:40:06,336 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-16 18:40:25,008 - BERTopic - Dimensionality - Completed ✓
2026-09-16 18:40:25,009 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-16 18:40:25,138 - BERTopic - Cluster - Completed ✓
2026-09-16 18:40:25,142 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-16 18:40:30,852 - BERTopic - Representation - Completed ✓


In [7]:
# show topic dataframe
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3422,-1_수시_ac_www_kr,"[수시, ac, www, kr, 정시, 모집, 전형, 수능, 히딩크, 논술, 면접,...",[복수 지원 처리 대학입시 수험생 자신 적성 성적 대학 모집단위 선택 중요 모집 군...
1,0,332,0_20_예비_지정_외국어,"[20, 예비, 지정, 외국어, 탐구, 수리, 대학생, 신청, 과목, 세대, 번역,...",[생존 보장 가능 포기 폐기 프로그램 폐기 남북 북미 회담 의미 김준 정상 결정 실...
2,1,225,1_트럼프_회담_정상_북미,"[트럼프, 회담, 정상, 북미, 대통령, 협상, 도널드, 국무, 행정부, 친서, 바...",[입장 대미 수세 승부수 시진핑 방북 향배 주목 북한 국빈 방문 시진핑 중국 국가 ...
3,2,190,2_30_피지_기증_마을,"[30, 피지, 기증, 마을, 선발, 학년도, 소프트웨어, 탈락, 교육부, 공동체,...",[계획 비중 실적 교육인적 자원부 30일 법학전문대학원 로스쿨 설치 심사기준 교육과...
4,3,164,3_남북_대북_한반도_대화,"[남북, 대북, 한반도, 대화, 외교, 안보, 비핵화, 제재, 북한, 대남, 문재,...",[미정상회담 다자대화 선행 신뢰 조처 송민순 외교 통상부 장관 교수 문정인 연세대 ...
...,...,...,...,...,...
66,65,21,65_바이오_페미니즘_본고사_권리,"[바이오, 페미니즘, 본고사, 권리, 예배, 징계, 중독, 자유, 총회, 종교, 경...",[이강덕 포항 시장 유치 포스텍 의과 대학 바이오 산업 육성 광폭 행보 경북 포항시...
67,66,21,66_총장_신학_학번_61,"[총장, 신학, 학번, 61, 사모, 순환, 출신, 설립자, 하원, 대학원, 신대,...",[한동대 사태 소유 분쟁 경북 포항 한동대 총학생회 교수 학부모기도회 연석회의 김영...
68,67,21,67_자전거_종로_도로_광역,"[자전거, 종로, 도로, 광역, 전지, 기숙사, 퓨처, 서울시, 전용, 전공, 선발...",[무학 선발 고교 교육 화두 진로 과목 선택 대학들 계열 권장과목 선택과목 제공 과...
69,68,20,68_사학_석사_학사_31,"[사학, 석사, 학사, 31, 역임, 퓨처, 도박, 박사, 역사, 학회, 포스코, ...",[충북 공립 대안 학교 단재고 아이비 프로그램 토의 토론 수업 수행 논술 서술형 평...


In [8]:
# show 2d visualized topic like LDAvis
BERTopic_vis = topic_model.visualize_topics()
BERTopic_vis

In [9]:
# To visualize the hierarchy of topics
hierachy_vis = topic_model.visualize_hierarchy()
hierachy_vis

In [10]:
# To visualize the correlation of topics
heatmap_vis = topic_model.visualize_heatmap()
heatmap_vis

In [11]:
# To visualize the ranking of terms per topic
barchart_vis = topic_model.visualize_barchart(top_n_topics=20)
barchart_vis

In [12]:
# To visualize the Term score per topic 
rank_vis = topic_model.visualize_term_rank()
rank_vis

In [13]:
# get the topics_over_time result
topics_over_time = topic_model.topics_over_time(words, timestamp)

30it [00:08,  3.41it/s]


In [14]:
# show dataframe of topics_over_time result
topics_over_time.head(20)

,Topic,Words,Frequency,Timestamp
0,-1,"본고사, 경쟁, 학과, 입시, 미달",48,1995
1,0,"본고사, 특차, 미달, 과목, 정원",6,1995
2,2,"선린, 실력, nasa, 설립, 과학자",4,1995
3,4,"학점, 장벽, 은행, 교육, 기관",2,1995
4,6,"내신, 배정, 응시, 거주, 전문대",1,1995
5,8,"순결, 대학가, 개방, 가치관, 공론",1,1995
6,9,"본고사, 경쟁, 입시, 특차, 상위",5,1995
7,10,"모금, 사랑, 재활, 남편, 요즘",1,1995
8,12,"제한, 25, 명문, 표방, 상위",6,1995
9,13,"합격자, 등록, 확보, 기부금, 수익",5,1995


In [15]:
#show line plot of topics_over_time result
tot_vis = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=30)
tot_vis

In [ ]:
# save the result
BERTopic_vis.write_html("view/bertopic_vis.html")
hierachy_vis.write_html("view/hierachy_vis.html")
heatmap_vis.write_html("view/heatmap_vis.html")
barchart_vis.write_html("view/barchart_vis.html")
rank_vis.write_html("view/rank_vis.html")
tot_vis.write_html("view/tot_vis.html")

- In BERTopic model, the optimal number of topics is found as a result of HDBSCAN. Therefore, the methodology (Perplexity, Coherence) used in the existing topic modeling is not used separately.

- Unlike the DTM model, BERTopic includes time variables in the topic. This makes it possible to analyze continuous time fluctuations. for this reason, BERTopic is characterized by relatively accurate capture of topics even in rapidly changing data (ex, news, newspaper editorials, SNS, etc.).